# eg11 (v2 replay) — `%%ggb` construction with explicit labels; `getValueString` + `ggb.parser.tokenize` (v1) retired → the XML IR (C6)

v1 (`examples/eg11_slider.ipynb`, main): `%ggblab ggb '{cmds}'` with relative references (`_1`, `_`, `__`), then `getValueString('l1')` → `ggb.parser.tokenize` to read the points of an intersection list.

v2: the same construction written with explicit labels (relative references are v1 magic; in v2 a bare identifier is a label); list members are named (`D = lst1(1)`) and read from the XML IR; the list length is a numeric object (`Length`) read by `value`. `{Tangent(…)}` stays a raw list definition (brace question, teacher) — the closed world does not see the head inside the braces.

In [1]:
import sys, time; sys.path.insert(0, '/Users/manabu/work/ggblab-replay')
from ggblab_extra.geometry_ir import element_irs, command_edges
import ggblab.host.html_host as H; H.DEPLOY = 'https://cdn.geogebra.org/apps/deployggb.js'
from ggblab import GeoGebra
%load_ext ggblab.ipymagic

In [2]:
g = GeoGebra(appName='suite', showToolBar=True, showAlgebraInput=True); g

In [3]:
%%ggb g
:const :new
A = (0, 0)
c = Circle(A, 1)
B = Point(c)
l = Line(A, B)
C = Point(l)
p = PerpendicularLine(C, l)
lst1 = {Intersect(c, p)}
D = lst1(1)
E = lst1(2)
t1 = Polygon(A, D, E)
ang1 = Angle(A, D, C)
ang2 = Angle(A, E, C)
M = Midpoint(A, C)
c2 = Circle(M, A)
lst2 = {Intersect(c, c2)}
F = lst2(1)
G = lst2(2)
t2 = Polygon(A, C, G)
ang3 = Angle(A, G, C)
lst3 = {Tangent(C, c)}
u = lst3(1)
v = lst3(2)
nL1 = Length(lst1)
nL3 = Length(lst3)

In [4]:
labels = _; print('labels', labels)
print('errors:', g.errors())

## read the list members from the IR (no tokenizer): coordinates of D, E (Intersect list lst1) and F, G (lst2); the list lengths as numeric objects

In [5]:
irs = element_irs(g.xml(timeout=60))
for lab in ('B', 'C', 'D', 'E', 'F', 'G'):
    ir = irs[lab]; print(lab, ir.type, ir.coords, '<-', (ir.command.name, ir.command.inputs) if ir.command else 'free')
print('Length(lst1) =', g.value('nL1'), '| Length(lst3) =', g.value('nL3'), '| kinds', g.kind('lst1'), g.kind('u'))

## react to a drag of `C` along `l` (pull, one reaction per operation): D/E move, the list length is re-read

The browser-side driver does `SetCoords(C, 0.5, 0)` then `SetCoords(C, 2, 0)` (outside the circle: `Intersect(c, p)` becomes undefined → the members are undefined, `Length` stays 2 in GeoGebra's list semantics — the IR shows the coordinates as they are).

In [6]:
_ = g.events(); steps = 0; t_all = time.time()
while steps < 2 and time.time() - t_all < 100:
    t0 = time.time(); e = g.wait_update('C', timeout=40)
    if e is None: print('  no change within 40 s'); break
    irs = element_irs(g.xml(timeout=60)); steps += 1
    print(f'step {steps}: C moved (woke after {round(time.time() - t0, 2)} s): C', irs['C'].coords, '| D', irs['D'].coords, '| E', irs['E'].coords, '| Length(lst1) =', g.value('nL1'))

In [7]:
print('DONE', 'errors:', g.errors())